# 1. Setup, Dependencies & Reproducibility
Install required packages, import core libraries, set random seeds for reproducible runs, and select compute device.

In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 95.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.9 MB/s eta 0:00:00
Device configured: cuda


# 2. Specialized VSFC Ontology Knowledge Graph Engine
Initializes `VSFCOntologyEngine` tailored for 3-class sentiment analysis (Negative, Neutral, Positive) via deep semantic inference and polarity injection.

In [3]:
# Initialize VSFCOntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = VSFCOntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.9),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.4),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Loading RDF knowledge graph from: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_9.rdf
Loading Knowledge Graph for VSFC: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_9.rdf
Loaded 91005 triples.


# 3. Vector Distribution Inspection
Inspects 24-dimensional continuous feature vectors across representative test sentences and validates activation patterns.

In [4]:
COLS_24D = (
    ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
    + [
        "Pleas",
        "Unpleas",
        "Loss",
        "Danger",
        "Unpred",
        "Sudden",
        "GoalObs",
        "Unfair",
        "Other",
    ]
    + ["Int_Hi", "Int_Me", "Int_Lo"]
    + ["Pos", "Neg", "Neu"]
    + ["NegHit", "NegCnt"]
)


def inspect_vectors(texts, engine):
    """Build a structured inspection DataFrame for ontology feature vectors."""
    data = []
    for txt in texts:
        vec = engine.getvector(txt)
        if isinstance(vec, tuple):
            vec = vec[0]
        vec_rounded = np.round(vec, 2)
        row = {"Sentence": txt}
        for i, val in enumerate(vec_rounded):
            row[COLS_24D[i]] = val
        data.append(row)
    return pd.DataFrame(data)


sample_texts = [
    "Món này ngon tuyệt vời",
    "Thái độ phục vụ quá tệ hại",
    "Không ngon lắm",
    "Món ăn bình thường, không có gì đặc sắc",
    "Tôi rất thất vọng về cách làm việc",
    "Sợ quá đi mất",
    "Tôi chẳng thấy buồn cười gì cả",
]

df_vector_preview = inspect_vectors(sample_texts, ontengine)
display(
    df_vector_preview[
        ["Sentence", "Happiness", "Sadness", "Pos", "Neg", "Neu", "NegHit"]
    ]
)

,Sentence,Happiness,Sadness,Pos,Neg,Neu,NegHit
0,Món này ngon tuyệt vời,0.60,0.00,0.94,0.00,0.00,0.0
1,Thái độ phục vụ quá tệ hại,0.00,0.00,0.00,0.00,0.00,0.0
2,Không ngon lắm,0.00,0.00,0.00,0.00,0.00,1.0
3,"Món ăn bình thường, không có gì đặc sắc",0.00,0.00,0.00,0.00,0.74,1.0
4,Tôi rất thất vọng về cách làm việc,0.42,0.85,0.81,1.00,0.00,0.0
5,Sợ quá đi mất,0.00,0.00,0.00,0.98,0.00,0.0
6,Tôi chẳng thấy buồn cười gì cả,0.00,0.00,0.00,0.00,0.00,0.0


# 4. Dataset Loading & Preprocessing (UIT-VSFC)
Reads UIT-VSFC sentences and sentiment labels (0: Negative, 1: Neutral, 2: Positive), constructs PyTorch DataLoaders, and computes balanced class weights.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")

# Locate UIT-VSFC dataset directory
BASE_PATH = os.path.join(repo_root, 'data', 'vsfc')
print(f"Using dataset path: {BASE_PATH}")


def load_vsfc_folder(folder_name):
    """Load UIT-VSFC sentence and sentiment annotation files."""
    sent_path = f"{BASE_PATH}/{folder_name}/sents.txt"
    label_path = f"{BASE_PATH}/{folder_name}/sentiments.txt"
    with open(sent_path, "r", encoding="utf-8") as f:
        sentences = [line.strip() for line in f.readlines()]
    with open(label_path, "r", encoding="utf-8") as f:
        labels = [int(line.strip()) for line in f.readlines()]
    min_len = min(len(sentences), len(labels))
    return pd.DataFrame(
        {"Sentence": sentences[:min_len], "Sentiment": labels[:min_len]}
    )


df_train = load_vsfc_folder("train")
df_valid = load_vsfc_folder("dev")
df_test = load_vsfc_folder("test")

print(
    f"Dataset loaded: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}"
)


class VSFC_OntologyDataset(Dataset):

    def __init__(self, df, tokenizer, engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]["Sentence"])
        lbl = int(self.df.iloc[idx]["Sentiment"])

        enc = self.tokenizer(
            txt,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        vec = self.engine.getvector(txt)
        if isinstance(vec, tuple):
            vec = vec[0]

        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "ontology_features": torch.tensor(vec, dtype=torch.float),
            "labels": torch.tensor(lbl, dtype=torch.long),
        }


BATCH_SIZE = 32
train_loader = DataLoader(
    VSFC_OntologyDataset(df_train, tokenizer, ontengine),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader = DataLoader(
    VSFC_OntologyDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE
)
test_loader = DataLoader(
    VSFC_OntologyDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE
)

# Compute class weights
train_labels = df_train["Sentiment"].tolist()
cls_w = compute_class_weight(
    "balanced", classes=np.unique(train_labels), y=train_labels
)
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"Class Weights [Neg, Neu, Pos]: {weights_tensor.cpu().numpy()}")

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

Using dataset path: /kaggle/input/datasets/minkduck/uit-vsfc/UIT-VSFC
Dataset loaded: Train=11426, Dev=1583, Test=3166
Class Weights [Neg, Neu, Pos]: [0.71524256 8.315866   0.6749365 ]


# 5. Neural-Symbolic Model Architectures
Defines neural-symbolic architectures for 3-class sentiment analysis:
1. `ViSoBERT_Fusion`: Standard ViSoBERT backbone supporting baseline, concatenation, and gated projection.
2. `ViSoBERT_DeepOntology_V2`: Multi-layer bottleneck projection with LayerNorm and Residual Gating.
3. `OntologyConceptProjector` & `ViSoBERT_Ontology_CrossAttention` (OCA): Token-level multi-head cross-attention over projected concept tokens.

In [6]:
# Import proposed and benchmark model architectures from src.models
from src.models.final_comparison_models import (
    ViSoBERT_Baseline,
    ALDONAr_Model,
    KEAHT_Model,
    CombViSA_Model,
)

ONT_INPUT_DIM = 24
print('✅ Models loaded from src.models.final_comparison_models: ViSoBERT_Baseline, ALDONAr_Model, KEAHT_Model, CombViSA_Model')


# 6. Training & Evaluation Pipeline
Provides training utilities with validation monitoring, early stopping, and checkpoint restoration.

In [7]:
def evaluate(model, loader):
    """Evaluate accuracy and macro F1 score on a given DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for b in loader:
            ids = b["input_ids"].to(device)
            m = b["attention_mask"].to(device)
            o = b["ontology_features"].to(device)
            l = b["labels"].to(device)

            out = model(ids, m, o)
            _, pr = torch.max(out, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(l.cpu().numpy())

    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(model, name, epochs=15, patience=4):
    """Execute training loop with class-weighted cross-entropy and early stopping."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for ep in range(epochs):
        model.train()
        loss_acc = 0.0
        for b in train_loader:
            optimizer.zero_grad()
            ids = b["input_ids"].to(device)
            m = b["attention_mask"].to(device)
            o = b["ontology_features"].to(device)
            l = b["labels"].to(device)

            loss = criterion(model(ids, m, o), l)
            loss.backward()
            optimizer.step()
            loss_acc += loss.item()

        acc, f1 = evaluate(model, valid_loader)
        avg_loss = loss_acc / len(train_loader)
        print(
            f"Epoch {ep + 1:02d} | Loss: {avg_loss:.4f} | Val F1: {f1:.4f} | Val Acc: {acc:.4f}",
            end=" ",
        )

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            print("(New Best)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {ep + 1} epochs.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)

    return model

# 7. Model Training Across Configurations
Trains all comparative baseline and neural-symbolic models, including the proposed Ontology-Guided Cross-Attention architecture (OCA).

In [8]:
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
NUM_CLASSES = len(CLASS_NAMES)

# 1. Baseline ViSoBERT
m_baseline = ViSoBERT_Baseline(NUM_CLASSES)
m_baseline = train_experiment(
    m_baseline, "1. Baseline (ViSoBERT)", epochs=15, patience=3
)

# 2. ALDONAr
m_aldonar = ALDONAr_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM)
m_aldonar = train_experiment(
    m_aldonar, "2. ALDONAr (Dual-Gated Fusion)", epochs=15, patience=3
)

# 3. KEAHT
m_keaht = KEAHT_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM, n_heads=8)
m_keaht = train_experiment(
    m_keaht,
    "3. KEAHT (Knowledge-Enriched Attention)",
    epochs=15,
    patience=3,
)

# 4. CombViSA
m_combvisa = CombViSA_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM)
m_combvisa = train_experiment(
    m_combvisa,
    "4. CombViSA (ViSoBERT + BiLSTM + Ontology)",
    epochs=15,
    patience=3,
)

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. Baseline (ViSoBERT)


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 3. KEAHT (Knowledge-Enriched Attention)
Epoch 01 | Loss: 0.6581 | Val F1: 0.7475 | Val Acc: 0.8654 (New Best)
Epoch 02 | Loss: 0.4156 | Val F1: 0.7602 | Val Acc: 0.8749 (New Best)
Epoch 03 | Loss: 0.2779 | Val F1: 0.7977 | Val Acc: 0.9116 (New Best)
Epoch 04 | Loss: 0.1832 | Val F1: 0.7665 | Val Acc: 0.8958 (Patience: 1/3)
Epoch 05 | Loss: 0.1296 | Val F1: 0.8014 | Val Acc: 0.9147 (New Best)
Epoch 06 | Loss: 0.0824 | Val F1: 0.8014 | Val Acc: 0.9185 (Patience: 1/3)
Epoch 07 | Loss: 0.0736 | Val F1: 0.7958 | Val Acc: 0.9116 (Patience: 2/3)
Epoch 08 | Loss: 0.0399 | Val F1: 0.8038 | Val Acc: 0.9236 (New Best)
Epoch 09 | Loss: 0.0515 | Val F1: 0.8048 | Val Acc: 0.9185 (New Best)
Epoch 10 | Loss: 0.0449 | Val F1: 0.7953 | Val Acc: 0.9210 (Patience: 1/3)
Epoch 11 | Loss: 0.0343 | Val F1: 0.7946 | Val Acc: 0.9135 (Patience: 2/3)
Epoch 12 | Loss: 0.0384 | Val F1: 0.7909 | Val Acc: 0.9154 (Patience: 3/3)
Early stopping triggered after 12 epochs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 4. CombViSA (ViSoBERT + BiLSTM + Ontology)
Epoch 01 | Loss: 0.7152 | Val F1: 0.7187 | Val Acc: 0.8377 (New Best)
Epoch 02 | Loss: 0.4349 | Val F1: 0.7444 | Val Acc: 0.8686 (New Best)
Epoch 03 | Loss: 0.3042 | Val F1: 0.7869 | Val Acc: 0.8983 (New Best)
Epoch 04 | Loss: 0.2186 | Val F1: 0.8015 | Val Acc: 0.9141 (New Best)
Epoch 05 | Loss: 0.1501 | Val F1: 0.7677 | Val Acc: 0.8951 (Patience: 1/3)
Epoch 06 | Loss: 0.1241 | Val F1: 0.8027 | Val Acc: 0.9185 (New Best)
Epoch 07 | Loss: 0.0824 | Val F1: 0.7975 | Val Acc: 0.9179 (Patience: 1/3)
Epoch 08 | Loss: 0.0805 | Val F1: 0.7957 | Val Acc: 0.9217 (Patience: 2/3)
Epoch 09 | Loss: 0.0675 | Val F1: 0.7957 | Val Acc: 0.9204 (Patience: 3/3)
Early stopping triggered after 9 epochs.


# 8. Final Benchmark Evaluation & Classification Reports
Evaluates all trained architectures on the test set and outputs macro F1, overall accuracy, and class-level classification metrics for OCA.

In [9]:
def get_predictions(model, loader):
    """Extract ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# Collect predictions across models
y_true, p_base = get_predictions(m_baseline, test_loader)
_, p_aldonar = get_predictions(m_aldonar, test_loader)
_, p_keaht = get_predictions(m_keaht, test_loader)
_, p_comb = get_predictions(m_combvisa, test_loader)

models_preds = [
    ("Baseline (ViSoBERT)", p_base),
    ("ALDONAr", p_aldonar),
    ("KEAHT", p_keaht),
    ("CombViSA", p_comb),
]

summary_records = []
for name, preds in models_preds:
    summary_records.append(
        {
            "Model Architecture": name,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n==================== TEST SET BENCHMARK RESULTS ====================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print("\n========== CLASSIFICATION REPORT: ALDONAr ==========")
print(
    classification_report(y_true, p_aldonar, target_names=CLASS_NAMES, digits=4)
)

print("\n========== CLASSIFICATION REPORT: KEAHT ==========")
print(
    classification_report(y_true, p_keaht, target_names=CLASS_NAMES, digits=4)
)

print("\n========== CLASSIFICATION REPORT: CombViSA ==========")
print(
    classification_report(y_true, p_comb, target_names=CLASS_NAMES, digits=4)
)


==================== TEST SET BENCHMARK RESULTS ====================
 Model Architecture  Accuracy  Macro F1
Baseline (ViSoBERT)    0.8964    0.7669
            ALDONAr    0.8989    0.7622
              KEAHT    0.8901    0.7557
           CombViSA    0.8967    0.7582

========== CLASSIFICATION REPORT: ALDONAr ==========
              precision    recall  f1-score   support

    Negative     0.9206    0.9297    0.9251      1409
     Neutral     0.4371    0.4371    0.4371       167
    Positive     0.9283    0.9201    0.9242      1590

    accuracy                         0.8989      3166
   macro avg     0.7620    0.7623    0.7622      3166
weighted avg     0.8990    0.8989    0.8989      3166


========== CLASSIFICATION REPORT: KEAHT ==========
              precision    recall  f1-score   support

    Negative     0.8785    0.9546    0.9150      1409
     Neutral     0.4605    0.4192    0.4389       167
    Positive     0.9461    0.8824    0.9131      1590

    accuracy             

# 9. Neuro-Symbolic Rule-Based Post-Processing
Overrides uncertain neural predictions using high-confidence symbolic signals from the ontology feature representations.

In [10]:
class NeuroSymbolicReasoner:
    """Combines neural probability outputs with ontology polarities for rule-based overrides."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        ontology_threshold=0.55,
        model_conf_limit=0.85,
    ):
        self.model = (
            model.module if isinstance(model, nn.DataParallel) else model
        )
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        self.idx2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

    def infer(self, text):
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec = self.engine.getvector(text)
        if isinstance(vec, tuple):
            vec = vec[0]
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # Symbolic override signals: index 19 (Pos), index 20 (Neg)
        pos_signal = vec[19]
        neg_signal = vec[20]
        final_label = neural_label

        # Override low-confidence Neutral predictions when ontology detects strong polarity
        if pred_idx == 1:
            if (
                neg_signal > self.ont_threshold
                and confidence < self.model_conf_limit
            ):
                final_label = "Negative"
            elif (
                pos_signal > self.ont_threshold
                and confidence < self.model_conf_limit
            ):
                final_label = "Positive"

        return neural_label, final_label


reasoner = NeuroSymbolicReasoner(
    model=m_baseline,
    engine=ontengine,
    tokenizer=tokenizer,
    device=device,
    ontology_threshold=0.55,
    model_conf_limit=0.85,
)

y_true_ns = []
y_pred_base = []
y_pred_neuro = []
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}

for _, row in tqdm(
    df_test.iterrows(), total=len(df_test), desc="Neuro-Symbolic Inference"
):
    text = str(row["Sentence"])
    true_lbl = int(row["Sentiment"])

    base_lbl, neuro_lbl = reasoner.infer(text)
    y_true_ns.append(true_lbl)
    y_pred_base.append(label_map[base_lbl])
    y_pred_neuro.append(label_map[neuro_lbl])

acc_base = accuracy_score(y_true_ns, y_pred_base)
f1_base = f1_score(y_true_ns, y_pred_base, average="macro")
acc_neuro = accuracy_score(y_true_ns, y_pred_neuro)
f1_neuro = f1_score(y_true_ns, y_pred_neuro, average="macro")

df_rule_comparison = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Macro F1"],
        "Baseline (ViSoBERT Only)": [acc_base, f1_base],
        "Neuro-Symbolic (ViSoBERT + Rules)": [acc_neuro, f1_neuro],
        "Delta": [acc_neuro - acc_base, f1_neuro - f1_base],
    }
)

print("\n============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============")
print(df_rule_comparison.round(4).to_string(index=False))

total_overrides = sum(1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n)
print(
    f"\nTotal samples overridden: {total_overrides}/{len(df_test)} ({total_overrides / len(df_test) * 100:.2f}%)"
)

Neuro-Symbolic Inference:   0%|          | 0/3166 [00:00<?, ?it/s]


============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============
  Metric  Baseline (ViSoBERT Only)  Neuro-Symbolic (ViSoBERT + Rules)  Delta
Accuracy                    0.8964                             0.8992 0.0028
Macro F1                    0.7669                             0.7710 0.0041

Total samples overridden: 22/3166 (0.69%)


# 10. Statistical Significance Testing (Paired Permutation Test)
Calculates exact permutation p-values over 10,000 resamples to evaluate statistical significance ($p < 0.05$) across comparative models against OCA.

In [11]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    ("Baseline vs ALDONAr", p_base, p_aldonar),
    ("Baseline vs KEAHT", p_base, p_keaht),
    ("Baseline vs CombViSA", p_base, p_comb),
    ("ALDONAr vs KEAHT", p_aldonar, p_keaht),
    ("KEAHT vs CombViSA", p_keaht, p_comb),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
          Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)        Sig (F1)  p-val (Acc)       Sig (Acc)
 Baseline vs ALDONAr        0.7669        0.7622     0.0048     0.64114 Not Significant      0.59474 Not Significant
   Baseline vs KEAHT        0.7669        0.7557     0.0113     0.29567 Not Significant      0.14409 Not Significant
Baseline vs CombViSA        0.7669        0.7582     0.0087     0.40386 Not Significant      1.00000 Not Significant
    ALDONAr vs KEAHT        0.7622        0.7557     0.0065     0.52925 Not Significant      0.05599 Not Significant
   KEAHT vs CombViSA        0.7557        0.7582     0.0025     0.82992 Not Significant      0.16638 Not Significant

